# Token Code Usage Under Dataset Shift

This notebook compares tokenizer code usage between the in-distribution reference set, `imagenet`, and one or more out-of-distribution datasets. The core hypothesis is simple: if a model has a stable and expressive discrete vocabulary, its active-code set, usage distribution, and spatial deployment of codes should remain relatively consistent when the input distribution shifts.


## Experiment Setup

The heavy computation happens outside the notebook. For each model and dataset pair, the export script writes:

- `summary.json`: compact run-level statistics.
- `global_counts.npy`: total count for each code ID across the full dataset.
- `position_counts.npy`: count for each `(code_id, row, col)` token-grid location.
- `usage.csv`: one row per code with count, frequency, activity flag, and rank.

This notebook stays light: it loads those artifacts, compares `imagenet` against all available OOD datasets, and then drills down into one selected OOD dataset for distribution and positional analysis.


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_ROOT = Path('../code_usage_results')
MODEL_DIRS = {
    'LlamaGen': RESULTS_ROOT / 'llamagen',
    'VQGAN': RESULTS_ROOT / 'vqgan',
}
IN_DISTRIBUTION_DATASET = 'imagenet'
FOCUS_OOD_DATASET = 'imagenet_v2'
CODE_IDS_TO_VIEW = [0, 1]


In [ ]:
def load_run(run_dir):
    summary = json.loads((run_dir / 'summary.json').read_text())
    global_counts = np.load(run_dir / 'global_counts.npy')
    position_counts = np.load(run_dir / 'position_counts.npy')
    usage = pd.read_csv(run_dir / 'usage.csv')
    return {
        'summary': summary,
        'global_counts': global_counts,
        'position_counts': position_counts,
        'usage': usage,
    }


def discover_datasets():
    datasets_per_model = {}
    for model, model_dir in MODEL_DIRS.items():
        datasets_per_model[model] = sorted(path.name for path in model_dir.iterdir() if path.is_dir())
    shared = sorted(set.intersection(*[set(names) for names in datasets_per_model.values()]))
    return datasets_per_model, shared


def load_runs(dataset_names):
    runs = {}
    for model, model_dir in MODEL_DIRS.items():
        runs[model] = {}
        for dataset in dataset_names:
            runs[model][dataset] = load_run(model_dir / dataset)
    return runs


def active_mask(global_counts):
    return global_counts > 0


def position_entropy(position_counts):
    totals = position_counts.sum(axis=0, keepdims=True)
    probs = np.divide(position_counts, totals, out=np.zeros_like(position_counts, dtype=np.float64), where=totals > 0)
    log_probs = np.zeros_like(probs)
    mask = probs > 0
    log_probs[mask] = np.log(probs[mask])
    return -(probs * log_probs).sum(axis=0)


def normalized_code_position_map(position_counts, code_id):
    grid = position_counts[code_id].astype(np.float64)
    total = grid.sum()
    return grid / total if total > 0 else grid


def make_summary_rows(runs, dataset_names):
    rows = []
    for model, model_runs in runs.items():
        imagenet_active = model_runs[IN_DISTRIBUTION_DATASET]['summary']['active_codes']
        imagenet_mask = active_mask(model_runs[IN_DISTRIBUTION_DATASET]['global_counts'])
        for dataset in dataset_names:
            run = model_runs[dataset]
            summary = run['summary']
            dataset_mask = active_mask(run['global_counts'])
            rows.append({
                'model': model,
                'dataset': dataset,
                'n_images': summary['n_images'],
                'active_codes': summary['active_codes'],
                'dead_codes': summary['dead_codes'],
                'active_fraction_full': summary['active_fraction_full'],
                'active_fraction_vs_imagenet': summary['active_codes'] / imagenet_active,
                'active_overlap_with_imagenet': (dataset_mask & imagenet_mask).sum() / imagenet_active,
                'perplexity': summary['perplexity'],
                'top_10_mass': summary['top_10_mass'],
                'top_100_mass': summary['top_100_mass'],
                'top_500_mass': summary['top_500_mass'],
            })
    return pd.DataFrame(rows)


datasets_per_model, shared_datasets = discover_datasets()
assert IN_DISTRIBUTION_DATASET in shared_datasets, 'ImageNet baseline results are required.'
ood_datasets = [name for name in shared_datasets if name != IN_DISTRIBUTION_DATASET]
comparison_datasets = [IN_DISTRIBUTION_DATASET] + ood_datasets
runs = load_runs(comparison_datasets)
summary_df = make_summary_rows(runs, comparison_datasets)

print('Shared datasets:', shared_datasets)
summary_df.head()


## Quick Validation

Each exported run should satisfy one invariant: the global token count must match the sum over all position-specific counts, and both must equal `n_images * H * W`.


In [ ]:
validation_rows = []
for model, model_runs in runs.items():
    for dataset, run in model_runs.items():
        summary = run['summary']
        global_total = int(run['global_counts'].sum())
        position_total = int(run['position_counts'].sum())
        h, w = summary['token_grid_hw']
        expected_total = summary['n_images'] * h * w
        validation_rows.append({
            'model': model,
            'dataset': dataset,
            'global_total': global_total,
            'position_total': position_total,
            'expected_total': expected_total,
            'ok': global_total == position_total == expected_total,
        })

pd.DataFrame(validation_rows).sort_values(['dataset', 'model']).reset_index(drop=True)


## Run-Level Summary Across Datasets

This table is the main compact view. `active_fraction_vs_imagenet` answers the baseline-normalized question directly: for each model, what fraction of the ImageNet active vocabulary remains active on each OOD dataset?


In [ ]:
summary_df = summary_df.sort_values(['dataset', 'model']).reset_index(drop=True)
summary_df


## Active Code Count by Dataset

The first comparison is absolute utilization: how many codes become active on each dataset for each model.


In [ ]:
datasets = comparison_datasets
models = list(MODEL_DIRS)
x = np.arange(len(datasets))
width = 0.36

fig, ax = plt.subplots(figsize=(9, 4.5))
for offset, model, color in [(-width / 2, 'LlamaGen', 'C0'), (width / 2, 'VQGAN', 'C1')]:
    sub = summary_df[summary_df['model'] == model].set_index('dataset').loc[datasets]
    ax.bar(x + offset, sub['active_codes'], width=width, label=model, color=color, alpha=0.85)

ax.axhline(16384, linestyle='--', color='0.35', linewidth=1, label='Full codebook')
ax.set_xticks(x)
ax.set_xticklabels(datasets, rotation=20)
ax.set_ylabel('Active codes')
ax.set_title('Absolute active-code count by dataset')
ax.grid(axis='y', alpha=0.25)
ax.legend(frameon=False)
plt.show()


## Active Vocabulary Retention Relative to ImageNet

This is the more controlled view for the hypothesis. For each model, ImageNet is the reference point, and every OOD dataset is normalized by that model's own ImageNet active-code count.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
for offset, model, color in [(-width / 2, 'LlamaGen', 'C0'), (width / 2, 'VQGAN', 'C1')]:
    sub = summary_df[summary_df['model'] == model].set_index('dataset').loc[datasets]
    ax.bar(x + offset, sub['active_fraction_vs_imagenet'], width=width, label=model, color=color, alpha=0.85)

ax.axhline(1.0, linestyle='--', color='0.35', linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(datasets, rotation=20)
ax.set_ylabel('Active codes / ImageNet active codes')
ax.set_title('Retention of the ImageNet active vocabulary')
ax.grid(axis='y', alpha=0.25)
ax.legend(frameon=False)
plt.show()


## Focused Pair

The remaining plots focus on one OOD dataset at a time. Change `FOCUS_OOD_DATASET` in the setup cell when you want to switch from `imagenet_v2` to another target such as `sketch` or `objectnet`.


In [ ]:
assert FOCUS_OOD_DATASET in ood_datasets, f'{FOCUS_OOD_DATASET} is not available in the shared result set.'
focus_datasets = [IN_DISTRIBUTION_DATASET, FOCUS_OOD_DATASET]
focus_df = summary_df[summary_df['dataset'].isin(focus_datasets)].copy()
focus_df.sort_values(['model', 'dataset']).reset_index(drop=True)


## Sorted Usage Distribution

Within each model, compare the full ranked code-frequency curve on ImageNet against the selected OOD dataset. A steeper OOD curve means more concentrated code reuse under shift.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
for ax, model in zip(axes, MODEL_DIRS):
    for dataset, color in [(IN_DISTRIBUTION_DATASET, 'C2'), (FOCUS_OOD_DATASET, 'C3')]:
        counts = runs[model][dataset]['global_counts']
        freqs = np.sort(counts / counts.sum())[::-1]
        ax.plot(np.arange(1, len(freqs) + 1), freqs, label=dataset, linewidth=2, color=color)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_title(model)
    ax.set_xlabel('Code rank')
    ax.set_ylabel('Frequency')
    ax.grid(alpha=0.25, which='both')
    ax.legend(frameon=False)

fig.suptitle(f'Sorted code usage: ImageNet vs {FOCUS_OOD_DATASET}')
plt.show()


## Top-Code Mass

This compresses concentration into three numbers per dataset inside each model: how much of the total token mass is captured by the top 10, 100, and 500 most-used codes.


In [ ]:
mass_df = focus_df.melt(
    id_vars=['model', 'dataset'],
    value_vars=['top_10_mass', 'top_100_mass', 'top_500_mass'],
    var_name='bucket',
    value_name='mass',
)

bucket_order = ['top_10_mass', 'top_100_mass', 'top_500_mass']
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=True)
for ax, model in zip(axes, MODEL_DIRS):
    sub = mass_df[mass_df['model'] == model]
    for offset, dataset, color in [(-0.18, IN_DISTRIBUTION_DATASET, 'C2'), (0.18, FOCUS_OOD_DATASET, 'C3')]:
        rows = sub[sub['dataset'] == dataset].set_index('bucket').loc[bucket_order]
        ax.bar(np.arange(3) + offset, rows['mass'], width=0.36, label=dataset, color=color, alpha=0.85)
    ax.set_xticks(np.arange(3))
    ax.set_xticklabels(['Top 10', 'Top 100', 'Top 500'])
    ax.set_title(model)
    ax.set_ylabel('Fraction of all token usage')
    ax.grid(axis='y', alpha=0.25)
    ax.legend(frameon=False)

fig.suptitle(f'Usage concentration: ImageNet vs {FOCUS_OOD_DATASET}')
plt.show()


## Positional Entropy

For each token-grid location, this computes the entropy of `P(code | position)`. The question here is whether dataset shift changes the spatial diversity of the codes used at different parts of the image.


In [ ]:
entropy_grids = {
    model: {
        dataset: position_entropy(runs[model][dataset]['position_counts'])
        for dataset in focus_datasets
    }
    for model in MODEL_DIRS
}

fig, axes = plt.subplots(2, 2, figsize=(8, 8), constrained_layout=True)
vmin = min(grid.min() for model_grids in entropy_grids.values() for grid in model_grids.values())
vmax = max(grid.max() for model_grids in entropy_grids.values() for grid in model_grids.values())

for row, model in enumerate(MODEL_DIRS):
    for col, dataset in enumerate(focus_datasets):
        ax = axes[row, col]
        im = ax.imshow(entropy_grids[model][dataset], vmin=vmin, vmax=vmax, cmap='viridis')
        ax.set_title(f'{model} | {dataset}')
        ax.set_xticks([])
        ax.set_yticks([])

fig.colorbar(im, ax=axes, fraction=0.046, pad=0.04, label='Entropy')
fig.suptitle('Per-position code entropy')
plt.show()


## Where Individual Codes Appear

This is the direct positional view for specific code IDs. For each selected code, the heatmap shows the normalized spatial mass of that code on the token grid. If a code consistently appears in specific regions, the map will make that visible.


In [ ]:
for code_id in CODE_IDS_TO_VIEW:
    fig, axes = plt.subplots(2, 2, figsize=(8, 8), constrained_layout=True)
    vmax = 0.0
    maps = {}
    for model in MODEL_DIRS:
        maps[model] = {}
        for dataset in focus_datasets:
            grid = normalized_code_position_map(runs[model][dataset]['position_counts'], code_id)
            maps[model][dataset] = grid
            vmax = max(vmax, grid.max())

    for row, model in enumerate(MODEL_DIRS):
        for col, dataset in enumerate(focus_datasets):
            ax = axes[row, col]
            im = ax.imshow(maps[model][dataset], vmin=0.0, vmax=vmax, cmap='magma')
            ax.set_title(f'{model} | {dataset}')
            ax.set_xticks([])
            ax.set_yticks([])

    fig.colorbar(im, ax=axes, fraction=0.046, pad=0.04, label='Normalized position mass')
    fig.suptitle(f'Spatial deployment of code {code_id}')
    plt.show()


## Reading Guide

Use the notebook in this order:

1. Check the run-level summary across all shared datasets.
2. Look at `active_fraction_vs_imagenet` to see how much of each model's ImageNet vocabulary survives the shift.
3. Pick one OOD dataset with `FOCUS_OOD_DATASET` and inspect the ranked usage curves, concentration plots, and positional views.
4. When new datasets are added, place their exported files under `code_usage_results/<model>/<dataset>/` and rerun from the top.
